# 14 · Model Evaluation

**Purpose**: Deep test-set evaluation, error analysis, SHAP, segment
diagnostics.

| I/O | Path |
|-----|------|
| Input  | `data/processed/test.parquet` |
| Input  | `model/lgbm_atd_model.pkl` |
| Input  | `model/model_metadata.json` |
| Output | `data/processed/test_predictions.parquet` |

---
## 1 · Setup

In [ ]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as scipy_stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})
SEED = 42
SLA_THRESHOLD_MIN = 45

TEST_PATH      = Path('../data/processed/test.parquet')
MODEL_PATH     = Path('../model/lgbm_atd_model.pkl')
META_PATH      = Path('../model/model_metadata.json')
TEST_PRED_PATH = Path('../data/processed/test_predictions.parquet')

---
## 2 · Load Test Split, Model & Metadata

In [ ]:
with open(META_PATH, 'r', encoding='utf-8') as fh:
    meta = json.load(fh)

model        = joblib.load(MODEL_PATH)
FEATURE_COLS = meta['feature_cols']
CAT_FEATURES = meta['cat_features']
TARGET       = meta['target']

df_test = pd.read_parquet(TEST_PATH)
for col in CAT_FEATURES:
    if col in df_test.columns:
        df_test[col] = df_test[col].astype('category')

X_test = df_test[
    [c for c in FEATURE_COLS if c in df_test.columns]
]
y_test = df_test[TARGET].values

print(f'Test rows      : {len(df_test):,}')
print(f'Features       : {X_test.shape[1]}')
print(f'Model best iter: {meta["best_iteration"]}')

---
## 3 · Helper Functions

In [ ]:
def compute_mape(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    """MAPE excluding zero actuals."""
    mask = y_true > 0
    return float(
        np.mean(
            np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
        ) * 100
    )


def evaluate_model(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label: str = 'Model',
) -> dict:
    """Compute MAE, RMSE, MAPE, R², SLA-hit-rate."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = compute_mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    sla  = float(np.mean(y_pred <= SLA_THRESHOLD_MIN) * 100)
    return {
        'label': label,
        'MAE':   round(mae, 3),
        'RMSE':  round(rmse, 3),
        'MAPE':  round(mape, 2),
        'R2':    round(r2, 4),
        'SLA_%': round(sla, 2),
    }

---
## 4 · Final Test Metrics

In [ ]:
pred_test = np.clip(
    model.predict(X_test, num_iteration=meta['best_iteration']),
    a_min=0, a_max=None,
)
residuals = pred_test - y_test

result_test = evaluate_model(y_test, pred_test, 'LightGBM — Test')

# Compare against val metrics from metadata
val_row = {
    'label': 'LightGBM — Val (from metadata)',
    'MAE':   meta['val_mae'],
    'RMSE':  meta['val_rmse'],
    'MAPE':  meta['val_mape'],
    'R2':    meta['val_r2'],
    'SLA_%': None,
}
comparison = pd.DataFrame([val_row, result_test]).set_index('label')
print('── Val vs Test Performance ──')
print(comparison.to_string())

delta_mae = result_test['MAE'] - meta['val_mae']
print(
    f'\nTest MAE delta vs Val: {delta_mae:+.3f} min '
    f'({"overfit signal" if delta_mae > 1 else "no overfit"})',
)

---
## 5 · Feature Importance (Native Gain)

In [ ]:
imp = pd.DataFrame({
    'feature': model.feature_name(),
    'gain':    model.feature_importance('gain'),
}).sort_values('gain', ascending=False)
imp['gain_pct'] = imp['gain'] / imp['gain'].sum() * 100

top20 = imp.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    top20['feature'][::-1],
    top20['gain_pct'][::-1],
    color='#06C167',
    edgecolor='none',
)
ax.set_title(
    'Top 20 Features — LightGBM Gain (%)',
    fontweight='bold',
)
ax.set_xlabel('% of Total Gain')
plt.tight_layout()
plt.show()

print(imp.head(10).to_string(index=False))

---
## 6 · SHAP Summary Plot (Top 20 Features)

In [ ]:
try:
    import shap

    # Sample up to 5 000 rows to keep SHAP fast
    rng = np.random.default_rng(SEED)
    sample_idx = rng.choice(
        len(X_test), size=min(5000, len(X_test)), replace=False
    )
    X_shap = X_test.iloc[sample_idx]

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)

    shap.summary_plot(
        shap_values,
        X_shap,
        max_display=20,
        show=True,
        plot_type='dot',
    )
except ImportError:
    print(
        'shap not installed — skipping SHAP plot.'
        '  Install with: pip install shap'
    )

---
## 7 · Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Residual histogram ────────────────────────────────────────────────────
axes[0].hist(residuals, bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='black', linewidth=1)
axes[0].axvline(
    np.median(residuals), color='red', linestyle='--',
    linewidth=1.5,
    label=f'Median {np.median(residuals):.2f} min',
)
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Predicted − Actual (min)')
axes[0].legend()

# ── Q-Q plot ──────────────────────────────────────────────────────────────
sample = np.random.default_rng(SEED).choice(
    residuals, size=min(10_000, len(residuals)), replace=False
)
scipy_stats.probplot(sample, plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals')

# ── Residual vs predicted scatter ─────────────────────────────────────────
scatter_idx = np.random.default_rng(SEED).choice(
    len(pred_test), size=min(8000, len(pred_test)), replace=False
)
axes[2].scatter(
    pred_test[scatter_idx],
    residuals[scatter_idx],
    alpha=0.04, s=4, color='steelblue',
)
axes[2].axhline(0, color='red', linewidth=1)
axes[2].set_title('Residuals vs Predicted')
axes[2].set_xlabel('Predicted ATD (min)')
axes[2].set_ylabel('Predicted − Actual (min)')

plt.suptitle('Residual Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Residuals — mean: {residuals.mean():.3f}  '
      f'std: {residuals.std():.3f}  '
      f'skew: {scipy_stats.skew(residuals):.3f}')

---
## 8 · Segment Diagnostics (MAE by Group)

In [ ]:
df_eval = df_test.copy()
df_eval['ATD_predicted'] = pred_test
df_eval['residual']      = residuals
df_eval['abs_error']     = np.abs(residuals)

SEGMENT_COLS = [
    'territory', 'courier_flow', 'time_block',
    'geo_archetype', 'is_long_trip',
]
SEGMENT_COLS = [c for c in SEGMENT_COLS if c in df_eval.columns]

fig, axes = plt.subplots(
    1, len(SEGMENT_COLS),
    figsize=(4 * len(SEGMENT_COLS), 5),
)
if len(SEGMENT_COLS) == 1:
    axes = [axes]

for ax, seg in zip(axes, SEGMENT_COLS):
    seg_mae = (
        df_eval.groupby(seg, observed=True)['abs_error']
        .mean()
        .sort_values()
    )
    seg_mae.plot(
        kind='barh', ax=ax,
        color='#06C167', edgecolor='none',
    )
    ax.set_title(f'MAE by {seg}', fontweight='bold')
    ax.set_xlabel('MAE (min)')

plt.tight_layout()
plt.show()

# Printed tables
for seg in SEGMENT_COLS:
    tbl = (
        df_eval.groupby(seg, observed=True)
        .agg(
            n=('abs_error', 'count'),
            mae=('abs_error', 'mean'),
            median_residual=('residual', 'median'),
        )
        .round(2)
        .sort_values('mae', ascending=False)
    )
    print(f'\nMAE by {seg}:')
    print(tbl.to_string())

---
## 9 · Calibration Check — Decile Plot

In [ ]:
df_cal = pd.DataFrame({
    'actual':    y_test,
    'predicted': pred_test,
})
df_cal['decile'] = pd.qcut(
    df_cal['predicted'], q=10, labels=False, duplicates='drop'
)
cal_summary = (
    df_cal.groupby('decile')[['actual', 'predicted']]
    .mean()
    .round(2)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    cal_summary.index,
    cal_summary['actual'],
    marker='o', label='Mean actual ATD',
    color='steelblue', linewidth=2,
)
ax.plot(
    cal_summary.index,
    cal_summary['predicted'],
    marker='s', label='Mean predicted ATD',
    color='coral', linewidth=2, linestyle='--',
)
ax.set_title(
    'Calibration: Mean Predicted vs Mean Actual ATD by Decile',
    fontweight='bold',
)
ax.set_xlabel('Predicted ATD Decile')
ax.set_ylabel('ATD (min)')
ax.legend()
plt.tight_layout()
plt.show()

print(cal_summary.to_string())

---
## 10 · Save Test Predictions

In [ ]:
test_preds_df = pd.DataFrame({
    'workflow_uuid': df_test['workflow_uuid'].values,
    'ATD':           y_test,
    'ATD_predicted': pred_test,
    'residual':      residuals,
})
test_preds_df.to_parquet(
    TEST_PRED_PATH, index=False, engine='pyarrow'
)
print(f'Test predictions → {TEST_PRED_PATH}  ({len(test_preds_df):,} rows)')
print(
    f'Final test MAE  = {result_test["MAE"]:.3f} min  |  '
    f'R² = {result_test["R2"]:.4f}  |  '
    f'SLA hit = {result_test["SLA_%"]:.1f}%'
)